In [208]:
import pandas as pd
import glob
import string
import os
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

In [69]:
iso_codes = pd.read_csv('/content/full_iso_code_mapping.csv')
joshi_classes = pd.read_csv('/content/joshi_classes.csv')
iso_code = iso_codes[~iso_codes['ISO_639_1'].isna()]
iso_dict = dict(zip(iso_code.ISO_639_1, iso_code.ISO_639_3))

In [127]:
def map_iso(code):
  if len(code) == 2 and code in iso_dict:
      return iso_dict[code]
  return code

def calculate_f1(items, zero_division=0) -> float:
    """Compute F1 score for binary or multiclass classification."""
    golds, preds = zip(*items)
    preds = [pred.replace('__label__', '').split('_')[0] for pred in preds]
    golds = [gold.split('-')[0].split('_')[0] for gold in golds]

    return f1_score(golds, preds, average="macro", zero_division=zero_division)


In [64]:
def stats_by_resource_size(file_pattern, lang_col, dataset_name, save_dir="resource_stats"):
    os.makedirs(save_dir, exist_ok=True)

    all_class_groups = []
    all_lang_groups = []

    for file in glob.glob(file_pattern):
        model_name = os.path.basename(file).split('_')[0]

        # Load data
        df = pd.read_csv(file)

        # Normalize language codes
        if dataset_name == 'smol':
            df['lang_code'] = df[lang_col].str.split('-').str[0].str.split('_').str[0]
            df['lang_code'] = df['lang_code'].map(map_iso)
        elif dataset_name == 'flores':
            df.rename(columns={lang_col: 'lang_code'}, inplace=True)

        # Merge with Joshi classes
        df_classes = pd.merge(df, joshi_classes, on='lang_code', how='left')
        df_classes.drop(columns=[c for c in df_classes.columns if c.startswith('language')], inplace=True)

        # For glotlid and openlid, recalculate f1 if needed
        if model_name in ['glotlid', 'openlid']:
            df_classes['f1'] = df_classes.apply(lambda row: calculate_f1([(row['lang_code'], row['pred_lang'])]), axis=1)

        # Aggregate stats
        lang_group = df_classes.groupby(["lang_code", "class"]).agg(
            f1=("f1", "mean"),
            fpr=("fpr", "mean"),
            sample_size=("lang_code", "count")
        ).reset_index()
        lang_group["model"] = model_name
        lang_group["dataset"] = dataset_name

        class_group = lang_group.groupby("class").agg(
            avg_f1=("f1", "mean"),
            num_languages=("lang_code", "count"),
            total_samples=("sample_size", "sum")
        ).reset_index()
        class_group["model"] = model_name
        class_group["dataset"] = dataset_name

        # Save per model separately (optional)
        base_model = os.path.basename(file).replace(".csv", "")
        lang_group.to_csv(f"{save_dir}/{base_model}_lang_group.csv", index=False)
        class_group.to_csv(f"{save_dir}/{base_model}_class_group.csv", index=False)

        # Collect for merging later
        all_lang_groups.append(lang_group)
        all_class_groups.append(class_group)

    # Merge all results
    merged_lang_group = pd.concat(all_lang_groups).reset_index(drop=True)
    merged_class_group = pd.concat(all_class_groups).reset_index(drop=True)

    # Save merged results
    merged_lang_group.to_csv(f"{save_dir}/{dataset_name}_merged_lang_group.csv", index=False)
    merged_class_group.to_csv(f"{save_dir}/{dataset_name}_mmerged_class_group.csv", index=False)

    print(f"✅ All stats saved in {save_dir}")



In [65]:
langs = {
    'bloom': 'language_code',
    'common-voice': 'language_code',
    'flores': 'language',
    'tweetlid': 'lang_code',
    'smol': 'language'
}
files = ['bloom', 'flores', 'tweetlid', 'smol', 'common-voice']

In [68]:
stats_by_resource_size('/content/drive/MyDrive/COMP598 Project/lid_preds/smol/*csv', "language",
                       "flores", save_dir="resource_stats")

KeyError: 'class'

In [230]:
def compute_classwise_fpr(df, gold_col='lang_code', pred_col='pred_lang'):
    # 1. Get unique labels (important: sort to align)
    labels = sorted(set(df[gold_col]) | set(df[pred_col]))

    # 2. Build confusion matrix
    cm = confusion_matrix(df[gold_col], df[pred_col], labels=labels)

    # 3. Compute FPR per class
    fpr_per_class = {}
    for idx, lang in enumerate(labels):
        fp = cm[:, idx].sum() - cm[idx, idx]  # all instances predicted as lang, except correct ones
        tn = cm.sum() - (cm[idx, :].sum() + cm[:, idx].sum() - cm[idx, idx])  # all not-predicted as lang and not true lang
        if (fp + tn) > 0:
            fpr = fp / (fp + tn)
        else:
            fpr = 0.0
        fpr_per_class[lang] = fpr

    # 4. Return as a dataframe
    fpr_df = pd.DataFrame({
        'lang_code': list(fpr_per_class.keys()),
        'fpr': list(fpr_per_class.values())
    })

    return fpr_df

In [71]:
smol_files = glob.glob('/content/drive/MyDrive/COMP598 Project/lid_preds/smol/*csv')
smol_files

['/content/drive/MyDrive/COMP598 Project/lid_preds/smol/OpenLID_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/smol/glotlid_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/smol/cld3_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/smol/langid_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/smol/langdetect_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/smol/fasttext-language-identification_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/smol/franc_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/smol/glotlid_smol_classes.csv']

In [2]:
import pandas as pd

In [4]:
pd.read_csv('/content/drive/MyDrive/COMP598 Project/lid_preds/smol/langid_smol.csv')['language'].nunique()

111

In [313]:
flores = glob.glob('/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/*.csv')
flores

['/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/cld3_tweetlid_mono.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/langid_tweetlid_mono.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/fasttext-language-identification_tweetlid_mono.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/franc_tweetlid_mono.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/langdetect_tweetlid_mono.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/OpenLID_tweetlid_mono.csv',
 '/content/drive/MyDrive/COMP598 Project/lid_preds/tweetlid/glotlid_tweetlid_mono.csv']

In [312]:
our_files = glob.glob('/content/drive/MyDrive/COMP598 Project/our_model/*.csv')
our_files

['/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_tweetlid.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_bloom_stories.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_common_voice.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_flores.csv']

In [311]:
pd.read_csv(our_files[0])

,ID,tweet_id,tweet,lang_code,pred_lang,pred_prob,pred_name,gold_name,acc,f1,fpr
0,447713852135079936,NataliadeLimon9,Que viiiiiiiiiiida máis triste,gl,glg,0.998945,glg,glg,1.0,1.0,0.0
1,442066653946609665,mruiandre,"@RTP1, por que não dar valor à verdadeira músi...",pt,por,0.998750,por,por,1.0,1.0,0.0
2,448799023982718976,GuillemGirona,Molt fan de la gent que entra a la botiga a l'...,ca,cat,0.999999,cat,cat,1.0,1.0,0.0
3,443867621977759745,goraba89,@AsierPardavila y la brasileña no hace mas que...,es,spa,0.859413,spa,spa,1.0,1.0,0.0
4,446967855410872320,albagrawoosky7,toma esq lo sabia jajajajajaja real madrid - b...,es,spa,0.886373,spa,spa,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
16501,442395718410833921,DavidMontero_,@PGranadal 4 contando Blanes ;),es,ces,0.374428,ces,spa,0.0,0.0,1.0
16502,443033014432698368,albertinholugo,@mmmilucha gracias por compartir tu fashion se...,es,tgk,0.472362,tgk,spa,0.0,0.0,1.0
16503,445651379189977088,pablopeter7,"@Juanchuu17 gracias Capi, a ver que me dicen. ...",es,spa,0.430589,spa,spa,1.0,1.0,0.0
16504,440178030364090368,seagerfinancial,#Hoolahan hat-trick in 2nd half ????,en,jav,0.887932,jav,eng,0.0,0.0,1.0


In [319]:
for file in flores:
  df = pd.read_csv(file)
  # try:
  df['lang_code'] = df['lang_code'].str.split('-').str[0].str.split('_').str[0]
  # except:
  #   df['lang_code'] = df['language'].str.split('-').str[0].str.split('_').str[0]
  df['lang_code'] = df['lang_code'].map(map_iso)
  df['pred_lang'] = df['pred_lang'].str.replace('__label__', '').str.split('_').str[0].str.split('-').str[0]
  df['pred_lang'] = df['pred_lang'].map(map_iso)
  fpr = compute_classwise_fpr(df)
  model_name = os.path.basename(file).split('_')[0].replace('.csv', '')
  print(f"Model: {model_name}; FPR: {fpr['fpr'].mean()}")

Model: cld3; FPR: 0.004622699788145029
Model: langid; FPR: 0.004077747260888844
Model: fasttext-language-identification; FPR: 0.0024541793740463546
Model: franc; FPR: 0.0024157147736223127
Model: langdetect; FPR: 0.010019952032384177
Model: OpenLID; FPR: 0.002318864570936977
Model: glotlid; FPR: 0.0010354201642876482


In [310]:
# for file in our_files:
df = pd.read_csv(our_files[0])
# try:
df['lang_code'] = df['language_code'].str.split('-').str[0].str.split('_').str[0]
# except:
#   df['lang_code'] = df['language'].str.split('-').str[0].str.split('_').str[0]
df['lang_code'] = df['lang_code'].map(map_iso)
df['pred_lang'] = df['pred_lang'].str.replace('__label__', '').str.split('_').str[0].str.split('-').str[0]
df['pred_lang'] = df['pred_lang'].map(map_iso)
df['f1'] = df.apply(lambda row: calculate_f1([(row['lang_code'], row['pred_lang'])]), axis=1)
fpr = compute_classwise_fpr(df)
model_name = os.path.basename(our_files[0]).split('_')[-1].replace('.csv', '')
print(f"Model: {model_name}; F1: {df['f1'].mean()}")
print(f"Model: {model_name}; FPR: {fpr['fpr'].mean()}")

Model: flores; F1: 0.3565300285986654
Model: flores; FPR: 0.004417611578588099


In [244]:
def compute_micro_fpr(df, true_col="lang_code", pred_col="pred_lang"):
    total = len(df)
    false_positives = (df[true_col] != df[pred_col]).sum()
    true_negatives = total - false_positives

    micro_fpr = false_positives / total if total > 0 else 0
    return micro_fpr

In [268]:
for file in flores:
  df = pd.read_csv(file)
  df['lang_code'] = df['language'].str.split('-').str[0].str.split('_').str[0]
  df['lang_code'] = df['lang_code'].map(map_iso)
  df['pred_lang'] = df['pred_lang'].str.replace('__label__', '').str[0].str.split('_').str[0]
  df['pred_lang'] = df['pred_lang'].map(map_iso)
  fpr = compute_micro_fpr(df)
  model_name = os.path.basename(file).split('_')[0]
  print(f"Model: {model_name}; FPR: {fpr}")

Model: cld3; FPR: 1.0
Model: langdetect; FPR: 1.0
Model: franc; FPR: 1.0
Model: langid; FPR: 1.0
Model: fasttext-language-identification; FPR: 1.0
Model: OpenLID; FPR: 1.0
Model: glotlid; FPR: 1.0


In [260]:
df

,text,title,language_code,language,pred_lang,pred_prob,license,copyright,pageCount,bookInstanceId,bookLineage,pred_name,gold_name,accuracy,f1,precision,recall,fpr,lang_code
0,Bav gavq khail nail tsaol-aq al gaq neq khail ...,Neev geel aol nail jal tsil,aeu,Akeu,aeu,1.000006,cc-by-nc,"Copyright © 2022, Akheu literacy committee",11,2b824db7-f8c0-4cf2-8563-e0f4f669a576,056B6F11-4A6C-4942-B2BC-8861E62B03B3,Akeu,Akeu,1.0,1.0,1.0,1.0,0.0,aeu
1,"Teevq-n neq, yaq noil hail gaq aq bhal jaol na...",Thaoq kheeq nail ganq je,aeu,Akeu,aeu,1.000006,cc-by-nc,"Copyright © 2022, Akheu literacy committee",14,f3b9d239-b279-4f5b-9470-75d352221fd1,056B6F11-4A6C-4942-B2BC-8861E62B03B3,Akeu,Akeu,1.0,1.0,1.0,1.0,0.0,aeu
2,"Jul gu al gu bhai neq,miq tsiq miq ma teevq ga...",Naq thaoq nail ganq je,aeu,Akeu,aeu,0.999996,cc-by-nc,"Copyright © 2022, Akheu literacy committee",14,9ec81557-3208-4392-9c15-62b3a92fafb0,056B6F11-4A6C-4942-B2BC-8861E62B03B3,Akeu,Akeu,1.0,1.0,1.0,1.0,0.0,aeu
3,Teevq-n neq yaq yov gaq neq dai yal bhai dai y...,Miq tsiq miq ma nvq ma yaq,aeu,Akeu,aeu,1.000007,cc-by-nc,"Copyright © 2022, Akheu literacy committee",18,ef261a25-f202-4fce-b9f9-47f2eef16c00,056B6F11-4A6C-4942-B2BC-8861E62B03B3,Akeu,Akeu,1.0,1.0,1.0,1.0,0.0,aeu
4,Aq-u phao jaq laq neq aivq phaq yaq bhai gaq l...,Jaq laq khi phaq yaq nee geel yuq thaq nail ga...,aeu,Akeu,aeu,1.000005,cc-by-nc,"Copyright © 2022, Akheu literacy committee",28,fb350da4-28ff-4b11-8627-64e39d0f14c6,056B6F11-4A6C-4942-B2BC-8861E62B03B3,Akeu,Akeu,1.0,1.0,1.0,1.0,0.0,aeu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1044,老鼠种花生\n老鼠给花生浇水\n花生开花了\n老鼠看不到花生\n花生没有结果\n老鼠讲给兔子...,老鼠收花生,zho,Chinese,cmn,0.961885,cc-by-nc-sa,"Copyright © 2016, SIL International",15,9f7ff6db-7428-4adc-bc87-8f596fe53cd0,056B6F11-4A6C-4942-B2BC-8861E62B03B3,Mandarin Chinese,Chinese,0.0,0.0,0.0,0.0,1.0,zho
1045,有一只很大的狮子在树下睡觉了。那时候有一只很小的老鼠在他的肚子上跑过去。\n狮子觉得很痒，所...,Rơmung Dŭl Hăng Tơkuih,zho,Chinese,cmn,0.998566,cc-by-nc,"Copyright © 2022, Jơrai Bible Association.",15,8c488166-7ed2-4997-bb10-8752d705872d,"056B6F11-4A6C-4942-B2BC-8861E62B03B3,5d9cefd7-...",Mandarin Chinese,Chinese,0.0,0.0,0.0,0.0,1.0,zho
1046,"Ketika kerbau ini masih kecil, kerbau ini dipa...",KARABAU,zlm,Malay (individual language),zsm,0.897128,cc-by-nc-sa,"Copyright © 2021, Suausindak",13,e7efe252-6fb8-4349-88db-ec85addaa43b,8B8C1838-64E3-4989-93AB-251F960907FC,Standard Malay,Malay,0.0,0.0,0.0,0.0,1.0,zlm
1047,Kodwa emandulo umboko wendlovu wawum=shane fut...,Пилдин тентек баласы,zul,Zulu,zul,0.984097,cc-by,"Copyright © 2020, Бул чыгарма Creative Commons...",27,e44f2d54-4268-4542-ba9b-88a039f7468c,"056B6F11-4A6C-4942-B2BC-8861E62B03B3,7a08b3d1-...",Zulu,Zulu,1.0,1.0,1.0,1.0,0.0,zul


In [199]:
our_files = glob.glob('/content/drive/MyDrive/COMP598 Project/our_model/*.csv')
our_files

['/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram2_losssoftmax_tweetlid.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram2_losssoftmax_bloom_stories.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram2_losssoftmax_smol.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram2_losssoftmax_flores.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram2_losssoftmax_common_voice.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_tweetlid.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_bloom_stories.csv',
 '/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_smol.csv']

In [ ]:
all_class_groups = []
all_lang_groups = []

In [143]:
our_files[2]

'/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram2_losssoftmax_smol.csv'

In [133]:
model_name = os.path.basename(smol_files[6]).split('_')[0]

In [144]:
df = pd.read_csv(our_files[2])
df.sample(10)

,text,language,pred_lang,pred_prob,domain,pred_name,gold_name,acc,f1,fpr
60830,O amegile thata ka ba ba ileng ba swelwa ke ba...,tn,sot,0.993513,sentence,sot,tsn,0.0,0.0,1.0
21323,¡Esa sonrisa amplia e inocente hace que esta c...,es,spa,0.925408,sentence,spa,spa,1.0,1.0,0.0
14633,ཕྱི་ཟླ་9་པ་དང་བསྡུར་བ་ཡིན་ན་སྡེ་ཚན་གྱི་ཚོང་དེ་...,bo,swa,0.648201,sentence,swa,bod,0.0,0.0,1.0
18014,Mu 25 Vumbipati aabo bakalembwa kabalindila ba...,dov,xho,0.962380,sentence,xho,dov,0.0,0.0,1.0
63990,Ba ci jafe-jafe yi gënoon tar ci sunu àdduna d...,wo,wol,0.943148,sentence,wol,wol,1.0,1.0,0.0
36761,Gwel baton mar mondo igol chiwo mari kokalo e ...,luo,luo,0.999818,sentence,luo,luo,1.0,1.0,0.0
10757,Bushe amasambililo yobe ayaku sekondale yalibo...,bem,bem,0.984646,sentence,bem,bem,1.0,1.0,0.0
31912,اعم کور تصدیق زِ میون عدم تحفظژ بنیاد چھہؠ تسہ...,ks,kas,0.422778,sentence,kas,kas,1.0,1.0,0.0
2439,و زاد، الكراهب المملوكة مسبقا و المرخصة عندهم ...,aeb,ary,0.978622,sentence,ary,aeb,0.0,0.0,1.0
77429,په یو وخت کې، د جوان په نوم یو کوچنی هلک و چې ...,scl,pus,1.000009,document,pus,scl,0.0,0.0,1.0


In [263]:
df = pd.read_csv("/content/drive/MyDrive/COMP598 Project/our_model/model_lr0.1_dim200_ngram3_losssoftmax_smol_smol.csv")
df.sample(10)

,text,language,pred_lang,pred_prob,domain,pred_name,gold_name,acc,f1,fpr
36246,Makmana ni tije mapiyo mag weche yuto nyalo be...,luo,luo,1.000003,sentence,luo,luo,1.0,1.0,0.0
42397,Tsvake zvimbuto zvemuruwa womubeke pona panets...,ndc,zul,0.749612,sentence,zul,ndc,0.0,0.0,1.0
37351,A chunga dan tha tak mai khi i zawm anih chuan...,lus,lus,0.997982,sentence,lus,lus,1.0,1.0,0.0
32545,"दोयम तर्फ, इंसान सिंद जिंदिगी, खातर हज़रते इसा ...",ks-Deva,kas,0.977126,sentence,kas,ks-Deva,0.0,0.0,1.0
76839,जङ्गलका वा चिडियाखानाका जनावरहरू बढी खुसी हुन्...,ne,npi,0.896203,document,npi,nep,0.0,0.0,1.0
46350,"Gama kanaan, waa'ee wiirtuulee nyaata qopheess...",om,zul,0.369874,sentence,zul,orm,0.0,0.0,1.0
74,"Dr. Karter kafâ koqsooy, fereytaay, ibi koqsoo...",aa,zul,0.732277,sentence,zul,aar,0.0,0.0,1.0
77436,په یو وخت کې، د ماریسول په نوم یوه ښکلې ځوانه ...,scl,pus,0.999501,document,pus,scl,0.0,0.0,1.0
61905,Vanghana va hina vate na dizete... swo koka ri...,ts,tgk,0.484510,sentence,tgk,tso,0.0,0.0,1.0
56136,"Nako e itseng, ke ne ke le jwalo ka papakgae e...",st,sot,1.000008,sentence,sot,sot,1.0,1.0,0.0


In [215]:
df['f1'].mean()

np.float64(0.3565300285986654)

In [168]:
df['acc'].mean()

np.float64(0.332697807435653)

In [202]:
df.rename(columns={'lang_code': 'language'}, inplace=True)

In [264]:
df['lang_code'] = df['language'].str.split('-').str[0].str.split('_').str[0]
df['lang_code'] = df['lang_code'].map(map_iso)

In [225]:
df['pred_lang'] = df['pred_lang'].map(map_iso)
df.sample(10)

,text,language,pred_lang,pred_prob,domain,pred_name,gold_name,acc,f1,fpr,lang_code
2043,ضيوفنا يتناقشوا على كيفاش نوازنوا ما بين إنو ن...,aeb,acm,0.314257,sentence,acm,aeb,0.0,0.0,1.0,aeb
5166,Cuma mi pim utiye i kabedu 3 i america ma yo m...,alz,jav,0.307212,sentence,jav,alz,0.0,0.0,1.0,alz
48490,Lliw novelaqa huk niraqman tikrakun imatapas k...,qu,mlt,0.112469,sentence,mlt,que,0.0,0.0,1.0,que
36445,"Kokete ei odiechieng', miyo ji ariyo go wuotho...",luo,luo,0.992557,sentence,luo,luo,1.0,1.0,0.0,luo
34870,"Esalemi na persil ya kitoko, na citrouille ya ...",ln,lin,0.998525,sentence,lin,lin,1.0,1.0,0.0,lin
67474,Uma isilinganiso sangaphambilini singaphezu kw...,zu,zul,1.000009,sentence,zul,zul,1.0,1.0,0.0,zul
34720,"Ntango mosusu ezalaki elemba, mbala mosusu ya ...",ln,lin,0.998260,sentence,lin,lin,1.0,1.0,0.0,lin
81588,དངུལ་ལོར་གོང་འཕར་ཞེས་པ་ནི་སྤྱིར་ཚོགས་པའི་རིན་ག...,xsr-Tibt,bod,0.906377,document,bod,xsr-Tibt,0.0,0.0,1.0,xsr
69559,जातिवाद भारत म एक सामाजिक व्यवस्था स जिस म लोग...,bgq,hin,0.485823,document,hin,bgq,0.0,0.0,1.0,bgq
47839,To dey bring war kriminals to justis dey bring...,pcm,tgk,0.958288,sentence,tgk,pcm,0.0,0.0,1.0,pcm


In [266]:
# df_classes = pd.merge(df, joshi_classes, on='lang_code', how='left')
df['f1'] = df.apply(lambda row: calculate_f1([(row['lang_code'], row['pred_lang'])]), axis=1)
# df_classes.sample(10)

In [257]:
df['pred_lang']

,pred_lang
0,a
1,a
2,a
3,a
4,a
...,...
1044,c
1045,c
1046,z
1047,z


In [265]:
f1_score(df['lang_code'], df['pred_lang'], average='macro')

0.09771870911746774

In [259]:
f1_score(df['lang_code'], df['pred_lang'], average='macro')

0.4252752800754118

In [267]:
df['f1'].mean()

np.float64(0.308469782128967)

In [262]:
df['f1'].mean()

np.float64(0.8846520495710201)

In [231]:
fpr = compute_classwise_fpr(df_classes)
fpr

,lang_code,fpr
0,ace,0.000953
1,aeu,0.000000
2,afr,0.000955
3,ahk,0.000000
4,aph,0.000000
...,...,...
141,yid,0.000953
142,yom,0.000953
143,zho,0.000000
144,zlm,0.000000


In [232]:
fpr['fpr'].mean()

np.float64(0.004417611578588099)

In [178]:
df_classes['class'].value_counts()

,count
class,
1.0,22581
0.0,11929
2.0,7767
3.0,1726
5.0,1726


In [179]:
lang_group = df_classes.groupby(["lang_code", "class"]).agg(
            f1=("f1", "mean"),
            fpr=("fpr", "mean"),
            sample_size=("lang_code", "count")
        ).reset_index()
lang_group["model"] = model_name
lang_group["dataset"] = "smol"
lang_group

,lang_code,class,f1,fpr,sample_size,model,dataset
0,aar,0.0,0.000000,1.000000,863,franc,smol
1,afr,3.0,0.746234,0.253766,863,franc,smol
2,aka,1.0,0.952491,0.047509,863,franc,smol
3,amh,2.0,0.950174,0.049826,863,franc,smol
4,ara,5.0,0.066049,1.000000,863,franc,smol
5,arz,3.0,0.389340,0.610660,863,franc,smol
6,aym,1.0,0.000000,1.000000,863,franc,smol
7,bam,1.0,0.911935,0.088065,863,franc,smol
8,bfq,0.0,0.000000,1.000000,457,franc,smol
9,bgq,0.0,0.000000,1.000000,457,franc,smol


In [181]:
class_group = lang_group.groupby("class").agg(
            avg_f1=("f1", "mean"),
            num_languages=("lang_code", "count"),
            total_samples=("sample_size", "sum")
        ).reset_index()

class_group["model"] = "ours"
class_group["dataset"] = "smol"

class_group

,class,avg_f1,num_languages,total_samples,model,dataset
0,0.0,0.056356,19,11929,ours,smol
1,1.0,0.450240,26,22581,ours,smol
2,2.0,0.631775,9,7767,ours,smol
3,3.0,0.567787,2,1726,ours,smol
4,5.0,0.497683,2,1726,ours,smol


In [182]:
# Save per model separately (optional)
# base_model = os.path.basename(smol_files[0]).replace(".csv", "")
class_group.to_csv(f"/content/drive/MyDrive/COMP598 Project/resource_stats/ours_class_group.csv", index=False)

In [84]:
# Save per model separately (optional)
base_model = os.path.basename(smol_files[0]).replace(".csv", "")
lang_group.to_csv(f"/content/drive/MyDrive/COMP598 Project/resource_stats/{base_model}_lang_group.csv", index=False)
class_group.to_csv(f"/content/drive/MyDrive/COMP598 Project/resource_stats/{base_model}_class_group.csv", index=False)

In [ ]:


    for file in glob.glob(file_pattern):
        model_name = os.path.basename(file).split('_')[0]

        # Load data
        df = pd.read_csv(file)

        # Normalize language codes
        if dataset_name == 'smol':
            df['lang_code'] = df[lang_col].str.split('-').str[0].str.split('_').str[0]
            df['lang_code'] = df['lang_code'].map(map_iso)
        elif dataset_name == 'flores':
            df.rename(columns={lang_col: 'lang_code'}, inplace=True)

        # Merge with Joshi classes
        df_classes = pd.merge(df, joshi_classes, on='lang_code', how='left')
        df_classes.drop(columns=[c for c in df_classes.columns if c.startswith('language')], inplace=True)

        # For glotlid and openlid, recalculate f1 if needed
        if model_name in ['glotlid', 'openlid']:
            df_classes['f1'] = df_classes.apply(lambda row: calculate_f1([(row['lang_code'], row['pred_lang'])]), axis=1)

        # Aggregate stats
        lang_group = df_classes.groupby(["lang_code", "class"]).agg(
            f1=("f1", "mean"),
            fpr=("fpr", "mean"),
            sample_size=("lang_code", "count")
        ).reset_index()
        lang_group["model"] = model_name
        lang_group["dataset"] = dataset_name

        class_group = lang_group.groupby("class").agg(
            avg_f1=("f1", "mean"),
            num_languages=("lang_code", "count"),
            total_samples=("sample_size", "sum")
        ).reset_index()
        class_group["model"] = model_name
        class_group["dataset"] = dataset_name

        # Save per model separately (optional)
        base_model = os.path.basename(file).replace(".csv", "")
        lang_group.to_csv(f"{save_dir}/{base_model}_lang_group.csv", index=False)
        class_group.to_csv(f"{save_dir}/{base_model}_class_group.csv", index=False)

        # Collect for merging later
        all_lang_groups.append(lang_group)
        all_class_groups.append(class_group)

    # Merge all results
    merged_lang_group = pd.concat(all_lang_groups).reset_index(drop=True)
    merged_class_group = pd.concat(all_class_groups).reset_index(drop=True)

    # Save merged results
    merged_lang_group.to_csv(f"{save_dir}/{dataset_name}_merged_lang_group.csv", index=False)
    merged_class_group.to_csv(f"{save_dir}/{dataset_name}_mmerged_class_group.csv", index=False)

In [2]:
pd.read_csv('/content/drive/MyDrive/COMP598 Project/lid_preds/smol/cld3_smol.csv')

,text,language,pred_lang,pred_prob,domain,pred_name,gold_name,accuracy,f1,precision,recall,fpr
0,Inni mascaba kattaatak barsiyyi gexso innik yo...,aa,so,0.855945,sentence,Somali,Afar,0.0,0.0,0.0,0.0,1.0
1,Nee kah esserewaa marih cabsu sinni rooci cayl...,aa,ha,0.655619,sentence,Hausa,Afar,0.0,0.0,0.0,0.0,1.0
2,"Dayli caddoodah madabah beenik, gandabiil rami...",aa,so,0.886921,sentence,Somali,Afar,0.0,0.0,0.0,0.0,1.0
3,Uson sinni hadaf arciseeniih gersi mara uguugu...,aa,so,0.951369,sentence,Somali,Afar,0.0,0.0,0.0,0.0,1.0
4,Immay sissik maaliyya gexso dagom takku uwwi m...,aa,mt,0.452698,sentence,Maltese,Afar,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
82016,འཛིན་སྐྱོང་པས་མོ་ལ་ལ་ཀ་མང་པུ་འབིན་སིང་མཐའ་མ་ལ་...,xsr-Tibt,vi,0.665663,document,Vietnamese,Sherpa,0.0,0.0,0.0,0.0,1.0
82017,སྒྱུ་རྩལ་དང་རོལ་རྩེད་ནི་ང་ཚོའི་རིག་གཞུང་གི་ཆ་ཤ...,xsr-Tibt,mr,0.737925,document,Marathi,Sherpa,0.0,0.0,0.0,0.0,1.0
82018,རླངས་འཁོར་ཞིག་མྱུར་པོར་རྒྱུག་བཞིན་ཡོད་པས་ཚོད་འ...,xsr-Tibt,vi,0.615282,document,Vietnamese,Sherpa,0.0,0.0,0.0,0.0,1.0
82019,སེམས་ཁམས་བདེ་ཐང་གི་དཀའ་ངལ་ཡོད་མཁན་གྱི་མི་ཚོར་ར...,xsr-Tibt,ne,0.399903,document,Nepali,Sherpa,0.0,0.0,0.0,0.0,1.0


In [3]:
pd.read_csv('/content/drive/MyDrive/COMP598 Project/lid_preds/smol/glotlid_smol.csv')

,text,language,pred_lang,pred_prob,pred_name,gold_name,accuracy,f1,precision,recall,fpr
0,Inni mascaba kattaatak barsiyyi gexso innik yo...,aa,__label__gaz_Latn,0.679600,unknown,Afar,0.0,0.0,0.0,0.0,1.0
1,Nee kah esserewaa marih cabsu sinni rooci cayl...,aa,__label__zpv_Latn,0.673046,unknown,Afar,0.0,0.0,0.0,0.0,1.0
2,"Dayli caddoodah madabah beenik, gandabiil rami...",aa,__label__som_Latn,0.582228,unknown,Afar,0.0,0.0,0.0,0.0,1.0
3,Uson sinni hadaf arciseeniih gersi mara uguugu...,aa,__label__zaw_Latn,0.320231,unknown,Afar,0.0,0.0,0.0,0.0,1.0
4,Immay sissik maaliyya gexso dagom takku uwwi m...,aa,__label__wol_Latn,0.226522,unknown,Afar,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...
82016,འཛིན་སྐྱོང་པས་མོ་ལ་ལ་ཀ་མང་པུ་འབིན་སིང་མཐའ་མ་ལ་...,xsr-Tibt,__label__bod_Tibt,0.993903,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0
82017,སྒྱུ་རྩལ་དང་རོལ་རྩེད་ནི་ང་ཚོའི་རིག་གཞུང་གི་ཆ་ཤ...,xsr-Tibt,__label__bod_Tibt,0.994138,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0
82018,རླངས་འཁོར་ཞིག་མྱུར་པོར་རྒྱུག་བཞིན་ཡོད་པས་ཚོད་འ...,xsr-Tibt,__label__bod_Tibt,0.999283,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0
82019,སེམས་ཁམས་བདེ་ཐང་གི་དཀའ་ངལ་ཡོད་མཁན་གྱི་མི་ཚོར་ར...,xsr-Tibt,__label__bod_Tibt,0.988229,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0


In [60]:
pd.read_csv('/content/drive/MyDrive/COMP598 Project/lid_preds/flores/OpenLID_flores.csv')

,domain,topic,text,language,lang_script,pred_lang,pred_prob,pred_name,gold_name,accuracy,f1,precision,recall,fpr
0,wikinews,"disease, research, canada","""کامو جينو نا تيكويه عمو ٤ بولن ڽڠ هانا ديابيت...",ace,Arab,__label__ace_Arab,1.000007,unknown,Achinese,0.0,0.0,0.0,0.0,1.0
1,wikinews,"disease, research, canada",در. ايهود اور، ڤروفيسور کدوکترن بق يونيۏرسيتس ...,ace,Arab,__label__ace_Arab,0.999957,unknown,Achinese,0.0,0.0,0.0,0.0,1.0
2,wikinews,"disease, research, canada",لݢى لادوم اورڠ چاروڠ لاءينجيه، غوبڽن راݢو ڤکوه...,ace,Arab,__label__ace_Arab,1.000010,unknown,Achinese,0.0,0.0,0.0,0.0,1.0
3,wikinews,music,بق اورو سنين، سارا دانيوس، سيکريتاريس تتڤ کومي...,ace,Arab,__label__ace_Arab,1.000003,unknown,Achinese,0.0,0.0,0.0,0.0,1.0
4,wikinews,music,"دانيوس خن، ""جينو كامو هان مبوت سڤو. لون کا لون...",ace,Arab,__label__ace_Arab,1.000010,unknown,Achinese,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215551,wikivoyage,Natural wonders/Midnight sun,"Njengoba izindawo zingahlali abantu abaningi, ...",zul,Latn,__label__zul_Latn,1.000010,unknown,Zulu,0.0,0.0,0.0,0.0,1.0
215552,wikivoyage,Reason to travel/Working in Japan,Isiko lokusebenza laseJapani lilandelana kakhu...,zul,Latn,__label__zul_Latn,0.999944,unknown,Zulu,0.0,0.0,0.0,0.0,1.0
215553,wikivoyage,Reason to travel/Working in Japan,Amasudi ayindlela yokugqoka evamile ebhizinisi...,zul,Latn,__label__zul_Latn,0.999990,unknown,Zulu,0.0,0.0,0.0,0.0,1.0
215554,wikivoyage,Reason to travel/Working in Japan,Ukusebenza ngokubambisana endaweni yokusebenze...,zul,Latn,__label__zul_Latn,0.998916,unknown,Zulu,0.0,0.0,0.0,0.0,1.0


In [17]:
smol_df = pd.read_csv('/content/drive/MyDrive/COMP598 Project/lid_preds/smol/glotlid_smol.csv')
iso_codes = pd.read_csv('/content/full_iso_code_mapping.csv')
joshi_classes = pd.read_csv('/content/joshi_classes.csv')

In [19]:
iso_codes = pd.read_csv('/content/full_iso_code_mapping.csv')
joshi_classes = pd.read_csv('/content/joshi_classes.csv')
iso_code = iso_codes[~iso_codes['ISO_639_1'].isna()]
iso_dict = dict(zip(iso_code.ISO_639_1, iso_code.ISO_639_3))

In [ ]:

def stats_by_resource_size(filepath, lang_col, dataset_name, model_name):
  for file in glob.glob(filepath):
    df = pd.read_csv(file)
    if dataset_name == 'smol':
      df['lang_code'] = df[lang_col].str.split('-').str[0].str.split('_').str[0]
      df['lang_code'] = df['lang_code'].map(map_iso)
    if dataset_name == 'flores':
      df.rename(columns={lang_col: 'lang_code'}, inplace=True)

    df_classes = pd.merge(df, joshi_classes, on='lang_code', how='left')
    df_classes.drop(columns=['language_y'], inplace=True)
    if model_name == 'glotlid' or model_name == 'openlid':
      df_classes['f1'] = df_classes.apply(lambda row: calculate_f1([(row['lang_code'], row['pred_lang'])]), axis=1)

    resource_agg = smol_classes.groupby('class').agg(
    avg_f1=('f1', 'mean'),
    count=('lang_code', 'count')).reset_index()

    lang_group = smol_classes.groupby(["lang_code", "class"]).agg(
    f1=("f1", "mean"),
    fpr=("fpr", "mean"),
    sample_size=("lang_code", "count")).reset_index()

    class_group = lang_group.groupby("class").agg({
        "f1": "mean",
        "lang_code": "count",  # number of unique languages per class
        "sample_size": "sum"}).rename(columns={"lang_code": "num_languages"}).reset_index()

In [23]:
smol_df['lang_code'] = smol_df['language'].str.split('-').str[0].str.split('_').str[0]

In [24]:
def map_iso(code):
    if len(code) == 2 and code in iso_dict:
        return iso_dict[code]
    return code

smol_df['lang_code'] = smol_df['lang_code'].map(map_iso)

In [25]:
smol_df

,text,language,pred_lang,pred_prob,pred_name,gold_name,accuracy,f1,precision,recall,fpr,lang_code
0,Inni mascaba kattaatak barsiyyi gexso innik yo...,aa,__label__gaz_Latn,0.679600,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar
1,Nee kah esserewaa marih cabsu sinni rooci cayl...,aa,__label__zpv_Latn,0.673046,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar
2,"Dayli caddoodah madabah beenik, gandabiil rami...",aa,__label__som_Latn,0.582228,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar
3,Uson sinni hadaf arciseeniih gersi mara uguugu...,aa,__label__zaw_Latn,0.320231,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar
4,Immay sissik maaliyya gexso dagom takku uwwi m...,aa,__label__wol_Latn,0.226522,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar
...,...,...,...,...,...,...,...,...,...,...,...,...
82016,འཛིན་སྐྱོང་པས་མོ་ལ་ལ་ཀ་མང་པུ་འབིན་སིང་མཐའ་མ་ལ་...,xsr-Tibt,__label__bod_Tibt,0.993903,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr
82017,སྒྱུ་རྩལ་དང་རོལ་རྩེད་ནི་ང་ཚོའི་རིག་གཞུང་གི་ཆ་ཤ...,xsr-Tibt,__label__bod_Tibt,0.994138,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr
82018,རླངས་འཁོར་ཞིག་མྱུར་པོར་རྒྱུག་བཞིན་ཡོད་པས་ཚོད་འ...,xsr-Tibt,__label__bod_Tibt,0.999283,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr
82019,སེམས་ཁམས་བདེ་ཐང་གི་དཀའ་ངལ་ཡོད་མཁན་གྱི་མི་ཚོར་ར...,xsr-Tibt,__label__bod_Tibt,0.988229,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr


In [26]:
smol_df[smol_df['lang_code'].str.len() != 3]

,text,language,pred_lang,pred_prob,pred_name,gold_name,accuracy,f1,precision,recall,fpr,lang_code


In [27]:
joshi_classes

,language,class,lang_code
0,kàsim,0,NaN
1,mapoyo,0,mcg
2,yamdena,0,jmd
3,rikbaktsa,0,rkb
4,belorussian,0,NaN
...,...,...,...
2480,german,5,deu
2481,japanese,5,jpn
2482,french,5,fra
2483,arabic,5,ara


In [30]:
smol_classes = pd.merge(smol_df, joshi_classes, on='lang_code', how='left')
smol_classes.drop(columns=['language_y'], inplace=True)
smol_classes.rename({'language_x': 'language'}, inplace=True)
smol_classes

,text,language_x,pred_lang,pred_prob,pred_name,gold_name,accuracy,f1,precision,recall,fpr,lang_code,class
0,Inni mascaba kattaatak barsiyyi gexso innik yo...,aa,__label__gaz_Latn,0.679600,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
1,Nee kah esserewaa marih cabsu sinni rooci cayl...,aa,__label__zpv_Latn,0.673046,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
2,"Dayli caddoodah madabah beenik, gandabiil rami...",aa,__label__som_Latn,0.582228,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
3,Uson sinni hadaf arciseeniih gersi mara uguugu...,aa,__label__zaw_Latn,0.320231,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
4,Immay sissik maaliyya gexso dagom takku uwwi m...,aa,__label__wol_Latn,0.226522,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
82016,འཛིན་སྐྱོང་པས་མོ་ལ་ལ་ཀ་མང་པུ་འབིན་སིང་མཐའ་མ་ལ་...,xsr-Tibt,__label__bod_Tibt,0.993903,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0
82017,སྒྱུ་རྩལ་དང་རོལ་རྩེད་ནི་ང་ཚོའི་རིག་གཞུང་གི་ཆ་ཤ...,xsr-Tibt,__label__bod_Tibt,0.994138,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0
82018,རླངས་འཁོར་ཞིག་མྱུར་པོར་རྒྱུག་བཞིན་ཡོད་པས་ཚོད་འ...,xsr-Tibt,__label__bod_Tibt,0.999283,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0
82019,སེམས་ཁམས་བདེ་ཐང་གི་དཀའ་ངལ་ཡོད་མཁན་གྱི་མི་ཚོར་ར...,xsr-Tibt,__label__bod_Tibt,0.988229,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0


In [31]:
smol_classes.to_csv('/content/drive/MyDrive/COMP598 Project/lid_preds/smol/glotlid_smol_classes.csv', index=False)

In [35]:
from sklearn.metrics import f1_score, accuracy_score

In [38]:
def calculate_f1(items, zero_division=0) -> float:
    """Compute F1 score for binary or multiclass classification."""
    golds, preds = zip(*items)
    preds = [pred.replace('__label__', '').split('_')[0] for pred in preds]

    return f1_score(golds, preds, average="macro", zero_division=zero_division)

In [39]:
smol_classes['f1'] = smol_classes.apply(lambda row: calculate_f1([(row['lang_code'], row['pred_lang'])]), axis=1)
smol_classes

,text,language_x,pred_lang,pred_prob,pred_name,gold_name,accuracy,f1,precision,recall,fpr,lang_code,class
0,Inni mascaba kattaatak barsiyyi gexso innik yo...,aa,__label__gaz_Latn,0.679600,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
1,Nee kah esserewaa marih cabsu sinni rooci cayl...,aa,__label__zpv_Latn,0.673046,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
2,"Dayli caddoodah madabah beenik, gandabiil rami...",aa,__label__som_Latn,0.582228,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
3,Uson sinni hadaf arciseeniih gersi mara uguugu...,aa,__label__zaw_Latn,0.320231,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
4,Immay sissik maaliyya gexso dagom takku uwwi m...,aa,__label__wol_Latn,0.226522,unknown,Afar,0.0,0.0,0.0,0.0,1.0,aar,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
82016,འཛིན་སྐྱོང་པས་མོ་ལ་ལ་ཀ་མང་པུ་འབིན་སིང་མཐའ་མ་ལ་...,xsr-Tibt,__label__bod_Tibt,0.993903,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0
82017,སྒྱུ་རྩལ་དང་རོལ་རྩེད་ནི་ང་ཚོའི་རིག་གཞུང་གི་ཆ་ཤ...,xsr-Tibt,__label__bod_Tibt,0.994138,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0
82018,རླངས་འཁོར་ཞིག་མྱུར་པོར་རྒྱུག་བཞིན་ཡོད་པས་ཚོད་འ...,xsr-Tibt,__label__bod_Tibt,0.999283,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0
82019,སེམས་ཁམས་བདེ་ཐང་གི་དཀའ་ངལ་ཡོད་མཁན་གྱི་མི་ཚོར་ར...,xsr-Tibt,__label__bod_Tibt,0.988229,unknown,Sherpa,0.0,0.0,0.0,0.0,1.0,xsr,0.0


In [46]:
smol_classes[smol_classes['lang_code'].isna()]

,text,language_x,pred_lang,pred_prob,pred_name,gold_name,accuracy,f1,precision,recall,fpr,lang_code,class


In [44]:
resource_agg = smol_classes.groupby('class').agg(
    avg_f1=('f1', 'mean'),
    count=('lang_code', 'count')
).reset_index()

resource_agg

,class,avg_f1,count
0,0.0,0.532652,11929
1,1.0,0.652850,22581
2,2.0,0.954938,7767
3,3.0,0.726535,1726
4,5.0,0.500000,1726


In [52]:
lang_group = smol_classes.groupby(["lang_code", "class"]).agg(
    f1=("f1", "mean"),
    fpr=("fpr", "mean"),
    sample_size=("lang_code", "count")
).reset_index()

# Now group by Joshi class, averaging across languages
class_group = lang_group.groupby("class").agg({
    "f1": "mean",
    "lang_code": "count",  # number of unique languages per class
    "sample_size": "sum"
}).rename(columns={"lang_code": "num_languages"}).reset_index()
class_group

,class,f1,num_languages,sample_size
0,0.0,0.484012,19,11929
1,1.0,0.664955,26,22581
2,2.0,0.954938,9,7767
3,3.0,0.726535,2,1726
4,5.0,0.500000,2,1726


In [57]:
top5_per_class = (
    smol_classes.groupby(['class', 'lang_code'])
    .agg(f1_mean=("f1", "mean"), sample_data=("lang_code", "count"))
    .reset_index()
    .sort_values(['class', 'f1_mean'], ascending=[True, True])
    .groupby('class')
    .head(5)
)
top5_per_class

,class,lang_code,f1_mean,sample_data
0,0.0,aar,0.000000,863
1,0.0,bfq,0.000000,457
2,0.0,bgq,0.000000,457
5,0.0,grt,0.000000,457
7,0.0,kau,0.000000,863
19,1.0,aka,0.000000,863
20,1.0,aym,0.000000,863
23,1.0,din,0.000000,863
25,1.0,grn,0.000000,863
30,1.0,kon,0.000000,863


In [49]:
top5_per_class = (
    smol_classes.groupby(['class', 'lang_code'])
    .size()
    .reset_index(name='count')
    .sort_values(['class', 'count'], ascending=[True, False])
    .groupby('class')
    .head(5)
)
top5_per_class

,class,lang_code,count
0,0.0,aar,863
3,0.0,dyu,863
4,0.0,efi,863
7,0.0,kau,863
12,0.0,pcm,863
27,1.0,kas,1726
35,1.0,sat,1726
19,1.0,aka,863
20,1.0,aym,863
21,1.0,bam,863
